# 🤖 Google ADK + MCP — Multi-System Data Assistant

## Architecture Overview

```
┌─────────────────────────────────────────────────────────────────┐
│                    GRADIO CHAT FRONTEND                         │
└──────────────────────────┬──────────────────────────────────────┘
                           │
┌──────────────────────────▼──────────────────────────────────────┐
│              ADK ORCHESTRATOR AGENT (Claude Sonnet)             │
│         Intent Router → Tool Selector → Response Synthesizer   │
└────────┬──────────────────┬───────────────────────┬────────────┘
         │                  │                       │
┌────────▼──────┐  ┌────────▼──────┐  ┌────────────▼──────┐
│  MCP Server 1 │  │  MCP Server 2 │  │   MCP Server 3    │
│  Observability│  │ Business Data │  │  Action Server    │
│  (Read-Only)  │  │  (Masked PII) │  │  (Guardrailed)    │
└───────────────┘  └───────────────┘  └───────────────────┘
```

### Sections
1. **Setup & Dependencies**
2. **Shared Models & Config**
3. **MCP Server 1: Observability Server** (Logs, Metrics, Alerts)
4. **MCP Server 2: Business Data Server** (Orders, Transactions, Customers)
5. **MCP Server 3: Action Server** (Restart, Tickets, Scale)
6. **ADK Agent (Orchestrator)**
7. **Gradio Frontend (Chatbot UI)**
8. **Launch Application**

---
## 📦 Section 1: Setup & Dependencies

In [ ]:
# Install required packages
import subprocess, sys

packages = [
    "mcp",
    "gradio",
    "anthropic",
    "pydantic>=2.0",
    "anyio",
    "httpx",
    "python-dotenv",
    "asyncio",
]

for pkg in packages:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "--quiet", "--break-system-packages"],
        capture_output=True
    )

print("✅ All packages installed successfully")

In [ ]:
# Core imports
import os
import json
import asyncio
import logging
import threading
import time
import uuid
import random
from datetime import datetime, timedelta
from typing import Any, Optional
from dataclasses import dataclass, field, asdict
from enum import Enum

# MCP
from mcp.server.fastmcp import FastMCP

# Anthropic
import anthropic

# Gradio
import gradio as gr

# Logging setup
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("ADK-MCP-Assistant")

print("✅ Imports loaded")

---
## ⚙️ Section 2: Shared Models, Config & Mock Data

In [ ]:
# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

class Config:
    """Central configuration for the entire system."""

    # Anthropic API Key — set via env or paste here
    ANTHROPIC_API_KEY: str = os.getenv("ANTHROPIC_API_KEY", "YOUR_API_KEY_HERE")
    MODEL: str = "claude-sonnet-4-20250514"

    # MCP Server ports
    OBSERVABILITY_PORT: int = 8001
    BUSINESS_PORT: int = 8002
    ACTION_PORT: int = 8003

    # Action Server — allowed services allowlist
    ALLOWED_SERVICES: list[str] = field(default_factory=lambda: [
        "payment-service",
        "auth-service",
        "order-service",
        "notification-service",
        "analytics-service",
    ])

    # Limits
    MAX_LOG_ENTRIES: int = 50
    MAX_TRANSACTIONS: int = 100
    MAX_REPLICAS: int = 10

    # Ticket priorities
    VALID_PRIORITIES: list[str] = field(default_factory=lambda: ["low", "medium", "high", "critical"])


config = Config()
ALLOWED_SERVICES = [
    "payment-service", "auth-service", "order-service",
    "notification-service", "analytics-service"
]

print("✅ Config loaded")

In [ ]:
# ─────────────────────────────────────────────
# MOCK DATA STORE (In-memory enterprise data)
# ─────────────────────────────────────────────

class MockDataStore:
    """Simulates enterprise backend data stores."""

    # ── Observability Data ──
    SERVICES = ["payment-service", "auth-service", "order-service",
                "notification-service", "analytics-service"]

    LOG_LEVELS = ["INFO", "WARN", "ERROR", "DEBUG"]
    LOG_MESSAGES = {
        "payment-service": [
            "Payment gateway timeout after 30s",
            "Transaction rolled back: insufficient funds",
            "Stripe API rate limit exceeded",
            "Duplicate transaction detected, rejecting",
            "Payment processor connection pool exhausted",
            "Revenue dropped 23% in last hour — threshold alert",
        ],
        "auth-service": [
            "JWT token validation failed: expired",
            "Multiple failed login attempts from IP 192.168.1.x",
            "OAuth2 provider returned 503",
            "Session store Redis connection timeout",
        ],
        "order-service": [
            "Order state machine stuck in PENDING",
            "Inventory check failed: service unavailable",
            "Webhook delivery failed after 3 retries",
            "Order processing latency p99=4200ms (threshold: 2000ms)",
        ],
    }

    METRICS = {
        "payment-service": {
            "error_rate": [{"ts": -60, "val": 0.02}, {"ts": -30, "val": 0.08},
                           {"ts": -15, "val": 0.19}, {"ts": 0, "val": 0.23}],
            "latency_p99": [{"ts": -60, "val": 180}, {"ts": -30, "val": 450},
                            {"ts": -15, "val": 1200}, {"ts": 0, "val": 3100}],
            "throughput": [{"ts": -60, "val": 450}, {"ts": -30, "val": 380},
                           {"ts": -15, "val": 210}, {"ts": 0, "val": 95}],
        },
        "auth-service": {
            "error_rate": [{"ts": -60, "val": 0.01}, {"ts": 0, "val": 0.04}],
            "latency_p99": [{"ts": -60, "val": 45}, {"ts": 0, "val": 120}],
        },
        "order-service": {
            "error_rate": [{"ts": -60, "val": 0.03}, {"ts": 0, "val": 0.15}],
            "latency_p99": [{"ts": -60, "val": 800}, {"ts": 0, "val": 4200}],
        },
    }

    ALERTS = {
        "payment-service": [
            {"id": "ALT-001", "severity": "critical", "title": "Revenue threshold breached",
             "description": "Revenue down 23% vs 7-day avg. Started 14:30 UTC.",
             "triggered_at": "2025-01-15T14:30:00Z", "status": "firing"},
            {"id": "ALT-002", "severity": "high", "title": "Payment processor errors elevated",
             "description": "Error rate: 19% (threshold: 5%)",
             "triggered_at": "2025-01-15T14:25:00Z", "status": "firing"},
        ],
        "order-service": [
            {"id": "ALT-003", "severity": "high", "title": "Order processing latency critical",
             "description": "p99 latency at 4200ms, SLA breach imminent",
             "triggered_at": "2025-01-15T14:45:00Z", "status": "firing"},
        ],
        "auth-service": [],
        "notification-service": [],
        "analytics-service": [],
    }

    # ── Business Data ──
    CUSTOMERS = {
        "CUST-001": {
            "customer_id": "CUST-001",
            "name": "Acme Corporation",
            "email": "***@acme.com",        # masked
            "phone": "***-***-1234",         # masked
            "tier": "enterprise",
            "region": "US-WEST",
            "account_status": "active",
            "created_at": "2023-03-15",
            "total_lifetime_value": 485000,
        },
        "CUST-002": {
            "customer_id": "CUST-002",
            "name": "TechCorp Ltd",
            "email": "***@techcorp.io",
            "phone": "***-***-5678",
            "tier": "professional",
            "region": "EU-WEST",
            "account_status": "active",
            "created_at": "2024-01-10",
            "total_lifetime_value": 92000,
        },
        "CUST-003": {
            "customer_id": "CUST-003",
            "name": "Startup Inc",
            "email": "***@startup.dev",
            "phone": "***-***-9012",
            "tier": "starter",
            "region": "APAC",
            "account_status": "suspended",
            "created_at": "2024-06-20",
            "total_lifetime_value": 8500,
        },
    }

    ORDERS = {
        "CUST-001": [
            {"order_id": "ORD-8810", "product": "Enterprise Suite", "amount": 48000,
             "status": "failed", "created_at": "2025-01-15T14:20:00Z",
             "failure_reason": "Payment gateway timeout"},
            {"order_id": "ORD-8809", "product": "Enterprise Suite", "amount": 48000,
             "status": "completed", "created_at": "2025-01-14T10:00:00Z", "failure_reason": None},
            {"order_id": "ORD-8791", "product": "Add-on: Analytics", "amount": 5000,
             "status": "completed", "created_at": "2025-01-10T09:00:00Z", "failure_reason": None},
        ],
        "CUST-002": [
            {"order_id": "ORD-8812", "product": "Pro Plan", "amount": 2400,
             "status": "pending", "created_at": "2025-01-15T15:00:00Z", "failure_reason": None},
        ],
        "CUST-003": [
            {"order_id": "ORD-8800", "product": "Starter Plan", "amount": 299,
             "status": "failed", "created_at": "2025-01-15T13:45:00Z",
             "failure_reason": "Account suspended"},
        ],
    }

    TRANSACTIONS = [
        {"txn_id": "TXN-5001", "customer_id": "CUST-001", "amount": 48000,
         "status": "failed", "date": "2025-01-15", "gateway": "stripe", "error_code": "timeout"},
        {"txn_id": "TXN-5002", "customer_id": "CUST-003", "amount": 299,
         "status": "failed", "date": "2025-01-15", "gateway": "paypal", "error_code": "account_suspended"},
        {"txn_id": "TXN-4998", "customer_id": "CUST-001", "amount": 48000,
         "status": "completed", "date": "2025-01-14", "gateway": "stripe", "error_code": None},
        {"txn_id": "TXN-4997", "customer_id": "CUST-002", "amount": 2400,
         "status": "completed", "date": "2025-01-14", "gateway": "stripe", "error_code": None},
        {"txn_id": "TXN-4995", "customer_id": "CUST-001", "amount": 5000,
         "status": "completed", "date": "2025-01-10", "gateway": "stripe", "error_code": None},
        {"txn_id": "TXN-5003", "customer_id": "CUST-002", "amount": 2400,
         "status": "pending", "date": "2025-01-15", "gateway": "stripe", "error_code": None},
    ]

    # ── Action Audit Log ──
    ACTION_LOG: list[dict] = []


store = MockDataStore()
print("✅ Mock data store initialized")
print(f"  Services: {store.SERVICES}")
print(f"  Customers: {list(store.CUSTOMERS.keys())}")
print(f"  Total transactions: {len(store.TRANSACTIONS)}")

---
## 🔵 Section 3: MCP Server 1 — Observability Server

**Purpose:** Read-only access to logs, metrics, and alerts  
**Constraints:** Time validation, response size limits, no write access

In [ ]:
# ─────────────────────────────────────────────
# MCP SERVER 1: OBSERVABILITY SERVER
# ─────────────────────────────────────────────

observability_mcp = FastMCP(
    name="ObservabilityServer",
    instructions=(
        "Read-only observability server. Exposes logs, metrics, and alerts. "
        "All access is audited. No write operations permitted."
    )
)


def _validate_time_range(start_time: str, end_time: str) -> tuple[datetime, datetime]:
    """Validate and parse ISO 8601 time strings. Raises ValueError on invalid input."""
    try:
        start_dt = datetime.fromisoformat(start_time.replace("Z", "+00:00"))
        end_dt = datetime.fromisoformat(end_time.replace("Z", "+00:00"))
    except ValueError:
        raise ValueError(
            "Invalid time format. Use ISO 8601: YYYY-MM-DDTHH:MM:SSZ or YYYY-MM-DD"
        )
    if start_dt >= end_dt:
        raise ValueError("start_time must be before end_time")
    max_range = timedelta(days=7)
    if (end_dt - start_dt) > max_range:
        raise ValueError("Time range cannot exceed 7 days to prevent overload")
    return start_dt, end_dt


def _validate_service(service_name: str) -> str:
    """Validate service name against known services."""
    if service_name not in store.SERVICES:
        raise ValueError(
            f"Unknown service '{service_name}'. "
            f"Valid services: {store.SERVICES}"
        )
    return service_name


@observability_mcp.tool()
def get_logs(
    service_name: str,
    start_time: str,
    end_time: str,
) -> dict:
    """
    Retrieve structured log entries for a service within a time range.

    Args:
        service_name: The service to fetch logs for. Must be a known service.
        start_time: ISO 8601 start timestamp (e.g. '2025-01-15T00:00:00Z')
        end_time: ISO 8601 end timestamp (e.g. '2025-01-15T23:59:59Z')

    Returns:
        dict with 'logs' list (capped at MAX_LOG_ENTRIES) and metadata
    """
    try:
        service = _validate_service(service_name)
        start_dt, end_dt = _validate_time_range(start_time, end_time)
    except ValueError as e:
        return {"error": str(e), "status": "validation_failed"}

    # Generate realistic logs from mock data
    messages = store.LOG_MESSAGES.get(service, ["Service operating normally"])
    logs = []
    for i, msg in enumerate(messages[:config.MAX_LOG_ENTRIES]):
        level = "ERROR" if any(w in msg.lower() for w in ["error", "fail", "timeout", "drop", "breach"])\
                else "WARN" if any(w in msg.lower() for w in ["warn", "exceeded", "stuck"])\
                else "INFO"
        offset_minutes = (len(messages) - i) * 8
        ts = (end_dt - timedelta(minutes=offset_minutes)).strftime("%Y-%m-%dT%H:%M:%SZ")
        logs.append({
            "timestamp": ts,
            "level": level,
            "service": service,
            "message": msg,
            "trace_id": f"trace-{uuid.uuid4().hex[:8]}",
        })

    logger.info(f"[ObservabilityMCP] get_logs: service={service}, count={len(logs)}")
    return {
        "status": "ok",
        "service": service,
        "time_range": {"start": start_time, "end": end_time},
        "total_count": len(logs),
        "logs": logs,
    }


@observability_mcp.tool()
def get_metrics(
    service_name: str,
    metric_name: str,
    window: str,
) -> dict:
    """
    Retrieve time-series metrics for a service.

    Args:
        service_name: The service to fetch metrics for.
        metric_name: Metric to retrieve: 'error_rate', 'latency_p99', 'throughput'
        window: Time window: '15m', '30m', '1h', '6h', '24h'

    Returns:
        dict with metric series data and statistical summary
    """
    VALID_WINDOWS = {"15m": 15, "30m": 30, "1h": 60, "6h": 360, "24h": 1440}
    VALID_METRICS = ["error_rate", "latency_p99", "throughput"]

    if service_name not in store.SERVICES:
        return {"error": f"Unknown service: {service_name}", "status": "validation_failed"}
    if window not in VALID_WINDOWS:
        return {"error": f"Invalid window '{window}'. Valid: {list(VALID_WINDOWS.keys())}",
                "status": "validation_failed"}
    if metric_name not in VALID_METRICS:
        return {"error": f"Invalid metric '{metric_name}'. Valid: {VALID_METRICS}",
                "status": "validation_failed"}

    service_metrics = store.METRICS.get(service_name, {})
    series = service_metrics.get(metric_name, [{"ts": 0, "val": 0}])

    values = [p["val"] for p in series]
    current_val = values[-1] if values else 0
    baseline = values[0] if values else 0
    trend = "increasing" if current_val > baseline * 1.1 else \
            "decreasing" if current_val < baseline * 0.9 else "stable"

    logger.info(f"[ObservabilityMCP] get_metrics: service={service_name}, metric={metric_name}")
    return {
        "status": "ok",
        "service": service_name,
        "metric": metric_name,
        "window": window,
        "current_value": current_val,
        "baseline_value": baseline,
        "trend": trend,
        "unit": "%" if metric_name == "error_rate" else "ms" if metric_name == "latency_p99" else "req/s",
        "series": series,
        "summary": {
            "min": min(values),
            "max": max(values),
            "avg": round(sum(values) / len(values), 4),
        }
    }


@observability_mcp.tool()
def get_alerts(service_name: str) -> dict:
    """
    Retrieve active and recent alerts for a service.

    Args:
        service_name: The service to fetch alerts for.

    Returns:
        dict with list of alerts and severity summary
    """
    if service_name not in store.SERVICES:
        return {"error": f"Unknown service: {service_name}", "status": "validation_failed"}

    alerts = store.ALERTS.get(service_name, [])
    firing = [a for a in alerts if a["status"] == "firing"]

    logger.info(f"[ObservabilityMCP] get_alerts: service={service_name}, firing={len(firing)}")
    return {
        "status": "ok",
        "service": service_name,
        "total_alerts": len(alerts),
        "firing_count": len(firing),
        "severity_breakdown": {
            "critical": sum(1 for a in alerts if a["severity"] == "critical"),
            "high": sum(1 for a in alerts if a["severity"] == "high"),
            "medium": sum(1 for a in alerts if a["severity"] == "medium"),
        },
        "alerts": alerts,
    }


print("✅ MCP Server 1 (Observability) — tools registered:")
print("   • get_logs(service_name, start_time, end_time)")
print("   • get_metrics(service_name, metric_name, window)")
print("   • get_alerts(service_name)")

---
## 🟢 Section 4: MCP Server 2 — Business Data Server

**Purpose:** Structured enterprise data (orders, transactions, customer profiles)  
**Constraints:** PII masking, predefined filters only, mandatory field validation

In [ ]:
# ─────────────────────────────────────────────
# MCP SERVER 2: BUSINESS DATA SERVER
# ─────────────────────────────────────────────

business_mcp = FastMCP(
    name="BusinessDataServer",
    instructions=(
        "Read-only business data server. Exposes orders, transactions, and customer profiles. "
        "All PII is masked. Only predefined filters are supported."
    )
)


def _validate_customer_id(customer_id: str) -> str:
    """Validate customer_id format and existence."""
    if not customer_id or not isinstance(customer_id, str):
        raise ValueError("customer_id is required and must be a non-empty string")
    if not customer_id.startswith("CUST-"):
        raise ValueError("customer_id must follow format: CUST-XXXX")
    if customer_id not in store.CUSTOMERS:
        raise ValueError(f"Customer '{customer_id}' not found")
    return customer_id


def _validate_date_range(date_range: str) -> tuple[str, str]:
    """Parse and validate date range string 'YYYY-MM-DD:YYYY-MM-DD'."""
    if ":" not in date_range:
        raise ValueError("date_range must be 'YYYY-MM-DD:YYYY-MM-DD'")
    parts = date_range.split(":")
    if len(parts) != 2:
        raise ValueError("date_range must contain exactly one ':' separator")
    try:
        start = datetime.strptime(parts[0].strip(), "%Y-%m-%d")
        end = datetime.strptime(parts[1].strip(), "%Y-%m-%d")
    except ValueError:
        raise ValueError("date_range dates must be in YYYY-MM-DD format")
    if start > end:
        raise ValueError("Start date must be before end date")
    return parts[0].strip(), parts[1].strip()


@business_mcp.tool()
def get_orders(customer_id: str) -> dict:
    """
    Retrieve all orders for a specific customer.

    Args:
        customer_id: Customer identifier in format CUST-XXXX (required)

    Returns:
        dict with order list and summary statistics
    """
    try:
        cid = _validate_customer_id(customer_id)
    except ValueError as e:
        return {"error": str(e), "status": "validation_failed"}

    orders = store.ORDERS.get(cid, [])
    failed = [o for o in orders if o["status"] == "failed"]
    completed = [o for o in orders if o["status"] == "completed"]

    logger.info(f"[BusinessMCP] get_orders: customer={cid}, count={len(orders)}")
    return {
        "status": "ok",
        "customer_id": cid,
        "total_orders": len(orders),
        "summary": {
            "completed": len(completed),
            "failed": len(failed),
            "pending": len(orders) - len(failed) - len(completed),
            "total_value": sum(o["amount"] for o in completed),
            "failed_value": sum(o["amount"] for o in failed),
        },
        "orders": orders,
    }


@business_mcp.tool()
def get_transactions(date_range: str, status: str) -> dict:
    """
    Retrieve transactions filtered by date range and status.

    Args:
        date_range: Date range as 'YYYY-MM-DD:YYYY-MM-DD' (required)
        status: Filter by status — one of: 'all', 'completed', 'failed', 'pending' (required)

    Returns:
        dict with transaction list, totals, and failure breakdown
    """
    VALID_STATUSES = ["all", "completed", "failed", "pending"]

    if not date_range:
        return {"error": "date_range is required", "status": "validation_failed"}
    if status not in VALID_STATUSES:
        return {"error": f"Invalid status '{status}'. Valid: {VALID_STATUSES}",
                "status": "validation_failed"}

    try:
        start_str, end_str = _validate_date_range(date_range)
    except ValueError as e:
        return {"error": str(e), "status": "validation_failed"}

    # Filter by date range
    filtered = [
        t for t in store.TRANSACTIONS
        if start_str <= t["date"] <= end_str
    ]

    # Filter by status
    if status != "all":
        filtered = [t for t in filtered if t["status"] == status]

    # Apply limit
    filtered = filtered[:config.MAX_TRANSACTIONS]

    failed_txns = [t for t in filtered if t["status"] == "failed"]
    error_codes = {}
    for t in failed_txns:
        ec = t.get("error_code") or "unknown"
        error_codes[ec] = error_codes.get(ec, 0) + 1

    logger.info(f"[BusinessMCP] get_transactions: range={date_range}, status={status}, count={len(filtered)}")
    return {
        "status": "ok",
        "filters": {"date_range": date_range, "status": status},
        "total_count": len(filtered),
        "summary": {
            "total_volume": sum(t["amount"] for t in filtered),
            "failed_count": len(failed_txns),
            "failed_volume": sum(t["amount"] for t in failed_txns),
            "error_breakdown": error_codes,
        },
        "transactions": filtered,
    }


@business_mcp.tool()
def get_customer_profile(customer_id: str) -> dict:
    """
    Retrieve masked customer profile.

    Args:
        customer_id: Customer identifier in format CUST-XXXX (required)

    Returns:
        dict with customer profile (PII masked: email, phone hidden)
    """
    try:
        cid = _validate_customer_id(customer_id)
    except ValueError as e:
        return {"error": str(e), "status": "validation_failed"}

    profile = store.CUSTOMERS[cid].copy()
    # PII masking is already applied in the store, but enforce here too
    if "email" in profile and not profile["email"].startswith("***"):
        parts = profile["email"].split("@")
        profile["email"] = f"***@{parts[1]}" if len(parts) == 2 else "***@***.com"
    if "phone" in profile and not profile["phone"].startswith("***"):
        profile["phone"] = f"***-***-{profile['phone'][-4:]}"

    logger.info(f"[BusinessMCP] get_customer_profile: customer={cid}")
    return {"status": "ok", "profile": profile}


print("✅ MCP Server 2 (Business Data) — tools registered:")
print("   • get_orders(customer_id)")
print("   • get_transactions(date_range, status)")
print("   • get_customer_profile(customer_id)")

---
## 🔴 Section 5: MCP Server 3 — Action Server

**Purpose:** Controlled system actions with full guardrails  
**Constraints:** Allowlist, confirmation flag, immutable audit log, replicas cap

In [ ]:
# ─────────────────────────────────────────────
# MCP SERVER 3: ACTION SERVER
# ─────────────────────────────────────────────

action_mcp = FastMCP(
    name="ActionServer",
    instructions=(
        "Controlled action server. All operations require explicit confirmation. "
        "Only allowlisted services can be acted upon. Every action is audit-logged."
    )
)


def _audit_action(action: str, params: dict, result: str, success: bool) -> dict:
    """Record every action to the immutable audit log."""
    entry = {
        "audit_id": f"AUD-{uuid.uuid4().hex[:8].upper()}",
        "timestamp": datetime.utcnow().isoformat() + "Z",
        "action": action,
        "parameters": params,
        "result": result,
        "success": success,
    }
    store.ACTION_LOG.append(entry)
    logger.info(f"[ActionMCP][AUDIT] {entry['audit_id']} | {action} | success={success} | {result}")
    return entry


def _validate_service_allowlist(service_name: str) -> str:
    """Enforce service allowlist — only permitted services can be acted upon."""
    if not service_name or not isinstance(service_name, str):
        raise ValueError("service_name is required")
    if service_name not in ALLOWED_SERVICES:
        raise PermissionError(
            f"Service '{service_name}' is not on the allowed list. "
            f"Permitted services: {ALLOWED_SERVICES}"
        )
    return service_name


@action_mcp.tool()
def restart_service(service_name: str, confirmed: bool = False) -> dict:
    """
    Restart a system service. Requires explicit confirmation.

    Args:
        service_name: Name of the service to restart (must be on allowlist)
        confirmed: Must be True to execute. False returns a dry-run preview.

    Returns:
        dict with execution result and audit ID
    """
    params = {"service_name": service_name, "confirmed": confirmed}

    try:
        svc = _validate_service_allowlist(service_name)
    except (ValueError, PermissionError) as e:
        audit = _audit_action("restart_service", params, str(e), success=False)
        return {"status": "rejected", "error": str(e), "audit_id": audit["audit_id"]}

    if not confirmed:
        return {
            "status": "pending_confirmation",
            "message": f"DRY RUN: Would restart '{svc}'. Set confirmed=True to execute.",
            "impact": "Service will be unavailable for ~30-60 seconds during restart.",
            "service": svc,
        }

    # Simulated restart
    result_msg = f"Service '{svc}' restart initiated. ETA: 45s. New pod started successfully."
    audit = _audit_action("restart_service", params, result_msg, success=True)
    return {
        "status": "success",
        "message": result_msg,
        "audit_id": audit["audit_id"],
        "estimated_recovery_seconds": 45,
    }


@action_mcp.tool()
def create_ticket(
    title: str,
    description: str,
    priority: str,
    confirmed: bool = False,
) -> dict:
    """
    Create a support or incident ticket in the ticketing system.

    Args:
        title: Ticket title (required, max 200 chars)
        description: Detailed description (required, max 2000 chars)
        priority: Ticket priority — one of: 'low', 'medium', 'high', 'critical' (required)
        confirmed: Must be True to create. False returns a preview.

    Returns:
        dict with ticket ID and routing info
    """
    VALID_PRIORITIES = ["low", "medium", "high", "critical"]
    params = {"title": title, "priority": priority, "confirmed": confirmed}

    # Input validation
    if not title or not isinstance(title, str) or len(title.strip()) == 0:
        err = "title is required and cannot be empty"
        audit = _audit_action("create_ticket", params, err, success=False)
        return {"status": "validation_failed", "error": err, "audit_id": audit["audit_id"]}
    if len(title) > 200:
        err = "title must be 200 characters or fewer"
        audit = _audit_action("create_ticket", params, err, success=False)
        return {"status": "validation_failed", "error": err, "audit_id": audit["audit_id"]}
    if not description or len(description.strip()) == 0:
        err = "description is required and cannot be empty"
        audit = _audit_action("create_ticket", params, err, success=False)
        return {"status": "validation_failed", "error": err, "audit_id": audit["audit_id"]}
    if len(description) > 2000:
        err = "description must be 2000 characters or fewer"
        audit = _audit_action("create_ticket", params, err, success=False)
        return {"status": "validation_failed", "error": err, "audit_id": audit["audit_id"]}
    if priority not in VALID_PRIORITIES:
        err = f"Invalid priority '{priority}'. Valid: {VALID_PRIORITIES}"
        audit = _audit_action("create_ticket", params, err, success=False)
        return {"status": "validation_failed", "error": err, "audit_id": audit["audit_id"]}

    if not confirmed:
        return {
            "status": "pending_confirmation",
            "message": f"DRY RUN: Would create '{priority}' ticket titled '{title}'. Set confirmed=True to create.",
            "preview": {"title": title, "priority": priority, "description": description[:100] + "..."},
        }

    ticket_id = f"TKT-{random.randint(10000, 99999)}"
    team_routing = {
        "critical": "on-call-sre",
        "high": "platform-eng",
        "medium": "ops-team",
        "low": "backlog",
    }[priority]

    result_msg = f"Ticket {ticket_id} created | Priority: {priority} | Routed to: {team_routing}"
    audit = _audit_action("create_ticket", params, result_msg, success=True)
    return {
        "status": "success",
        "ticket_id": ticket_id,
        "priority": priority,
        "routed_to": team_routing,
        "message": result_msg,
        "audit_id": audit["audit_id"],
        "expected_response_sla": {
            "critical": "15 minutes",
            "high": "1 hour",
            "medium": "4 hours",
            "low": "next business day",
        }[priority],
    }


@action_mcp.tool()
def scale_service(
    service_name: str,
    replicas: int,
    confirmed: bool = False,
) -> dict:
    """
    Scale a service to the specified number of replicas.

    Args:
        service_name: Name of the service to scale (must be on allowlist)
        replicas: Target replica count. Must be between 1 and 10.
        confirmed: Must be True to execute. False returns a dry-run preview.

    Returns:
        dict with scaling result and audit ID
    """
    params = {"service_name": service_name, "replicas": replicas, "confirmed": confirmed}

    try:
        svc = _validate_service_allowlist(service_name)
    except (ValueError, PermissionError) as e:
        audit = _audit_action("scale_service", params, str(e), success=False)
        return {"status": "rejected", "error": str(e), "audit_id": audit["audit_id"]}

    if not isinstance(replicas, int) or replicas < 1 or replicas > config.MAX_REPLICAS:
        err = f"replicas must be an integer between 1 and {config.MAX_REPLICAS}"
        audit = _audit_action("scale_service", params, err, success=False)
        return {"status": "validation_failed", "error": err, "audit_id": audit["audit_id"]}

    if not confirmed:
        current = random.randint(2, 4)  # simulated current
        direction = "up" if replicas > current else "down"
        return {
            "status": "pending_confirmation",
            "message": f"DRY RUN: Would scale '{svc}' from {current} → {replicas} replicas ({direction}). Set confirmed=True to execute.",
            "current_replicas": current,
            "target_replicas": replicas,
            "direction": direction,
        }

    result_msg = f"Service '{svc}' scaled to {replicas} replicas. Kubernetes rollout initiated."
    audit = _audit_action("scale_service", params, result_msg, success=True)
    return {
        "status": "success",
        "service": svc,
        "replicas": replicas,
        "message": result_msg,
        "audit_id": audit["audit_id"],
        "estimated_rollout_seconds": replicas * 15,
    }


print("✅ MCP Server 3 (Action) — tools registered:")
print("   • restart_service(service_name, confirmed)")
print("   • create_ticket(title, description, priority, confirmed)")
print("   • scale_service(service_name, replicas, confirmed)")

---
## 🧠 Section 6: ADK Agent (Orchestrator)

The ADK Agent uses Claude Sonnet as the LLM backbone. It:
- Routes queries to appropriate MCP servers
- Calls tools based on intent
- Synthesizes multi-source answers
- Maintains conversation history

In [ ]:
# ─────────────────────────────────────────────
# TOOL REGISTRY
# Maps tool names → actual Python functions
# This is what the ADK Agent calls when
# the LLM selects a tool.
# ─────────────────────────────────────────────

TOOL_REGISTRY = {
    # Observability Server tools
    "get_logs": get_logs,
    "get_metrics": get_metrics,
    "get_alerts": get_alerts,
    # Business Data Server tools
    "get_orders": get_orders,
    "get_transactions": get_transactions,
    "get_customer_profile": get_customer_profile,
    # Action Server tools
    "restart_service": restart_service,
    "create_ticket": create_ticket,
    "scale_service": scale_service,
}

# Anthropic tool schemas (for LLM function-calling)
TOOL_SCHEMAS = [
    {
        "name": "get_logs",
        "description": "Retrieve structured log entries for a service within a time range. Use to diagnose errors, investigate incidents.",
        "input_schema": {
            "type": "object",
            "properties": {
                "service_name": {"type": "string", "description": "Service name e.g. payment-service, auth-service"},
                "start_time": {"type": "string", "description": "ISO 8601 start time e.g. 2025-01-15T00:00:00Z"},
                "end_time": {"type": "string", "description": "ISO 8601 end time e.g. 2025-01-15T23:59:59Z"},
            },
            "required": ["service_name", "start_time", "end_time"],
        },
    },
    {
        "name": "get_metrics",
        "description": "Retrieve time-series performance metrics. Use to check error rates, latency, throughput trends.",
        "input_schema": {
            "type": "object",
            "properties": {
                "service_name": {"type": "string"},
                "metric_name": {"type": "string", "enum": ["error_rate", "latency_p99", "throughput"],
                               "description": "Metric to retrieve"},
                "window": {"type": "string", "enum": ["15m", "30m", "1h", "6h", "24h"],
                          "description": "Time window"},
            },
            "required": ["service_name", "metric_name", "window"],
        },
    },
    {
        "name": "get_alerts",
        "description": "Retrieve active alerts and their severity for a service.",
        "input_schema": {
            "type": "object",
            "properties": {
                "service_name": {"type": "string"},
            },
            "required": ["service_name"],
        },
    },
    {
        "name": "get_orders",
        "description": "Retrieve orders for a specific customer. Use to investigate customer order issues.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string", "description": "Customer ID e.g. CUST-001"},
            },
            "required": ["customer_id"],
        },
    },
    {
        "name": "get_transactions",
        "description": "Retrieve transactions filtered by date range and status. Use to analyze revenue, failed payments, trends.",
        "input_schema": {
            "type": "object",
            "properties": {
                "date_range": {"type": "string", "description": "Date range as YYYY-MM-DD:YYYY-MM-DD e.g. 2025-01-14:2025-01-15"},
                "status": {"type": "string", "enum": ["all", "completed", "failed", "pending"]},
            },
            "required": ["date_range", "status"],
        },
    },
    {
        "name": "get_customer_profile",
        "description": "Retrieve a customer's profile (PII masked). Use to understand customer tier, region, status.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string"},
            },
            "required": ["customer_id"],
        },
    },
    {
        "name": "restart_service",
        "description": "Restart a system service. Requires explicit user confirmation (confirmed=True). Always do a dry-run first (confirmed=False) unless user explicitly says to proceed.",
        "input_schema": {
            "type": "object",
            "properties": {
                "service_name": {"type": "string", "description": f"Service to restart. Allowed: {ALLOWED_SERVICES}"},
                "confirmed": {"type": "boolean", "description": "Set True only if user explicitly confirmed the action"},
            },
            "required": ["service_name", "confirmed"],
        },
    },
    {
        "name": "create_ticket",
        "description": "Create an incident or support ticket. Shows preview unless confirmed=True.",
        "input_schema": {
            "type": "object",
            "properties": {
                "title": {"type": "string", "description": "Ticket title (max 200 chars)"},
                "description": {"type": "string", "description": "Detailed description (max 2000 chars)"},
                "priority": {"type": "string", "enum": ["low", "medium", "high", "critical"]},
                "confirmed": {"type": "boolean", "description": "Set True only if user explicitly confirmed"},
            },
            "required": ["title", "description", "priority", "confirmed"],
        },
    },
    {
        "name": "scale_service",
        "description": "Scale a service to a specific replica count (1-10). Shows preview unless confirmed=True.",
        "input_schema": {
            "type": "object",
            "properties": {
                "service_name": {"type": "string"},
                "replicas": {"type": "integer", "minimum": 1, "maximum": 10},
                "confirmed": {"type": "boolean"},
            },
            "required": ["service_name", "replicas", "confirmed"],
        },
    },
]

print(f"✅ Tool registry loaded: {len(TOOL_REGISTRY)} tools across 3 MCP servers")

In [ ]:
# ─────────────────────────────────────────────
# ADK ORCHESTRATOR AGENT
# ─────────────────────────────────────────────

SYSTEM_PROMPT = """
You are an Enterprise Data Assistant — an intelligent AI agent with access to three MCP servers:

🔵 OBSERVABILITY SERVER — Logs, metrics, alerts (read-only)
   Tools: get_logs, get_metrics, get_alerts
   Use for: Error investigations, performance analysis, incident detection

🟢 BUSINESS DATA SERVER — Orders, transactions, customer profiles (PII masked)
   Tools: get_orders, get_transactions, get_customer_profile
   Use for: Revenue analysis, order issues, customer investigations

🔴 ACTION SERVER — System actions with guardrails (requires confirmation)
   Tools: restart_service, create_ticket, scale_service
   Use for: Remediation actions, incident tickets, scaling
   ⚠️ CRITICAL: Always do a dry-run first (confirmed=False). Only set confirmed=True if the
   user EXPLICITLY says "yes", "confirm", "proceed", "do it", or similar confirmation.

BEHAVIOR GUIDELINES:
- For diagnostic questions (revenue drop, errors, etc.): Call multiple tools to correlate data
- Always show tool results clearly with relevant numbers and trends
- For action requests: Show the dry-run first, then ask for confirmation
- Use markdown formatting: headers, bold for key numbers, bullet points for lists
- Be precise and data-driven. Quote actual values from tool results.
- If a tool returns an error, explain it clearly and suggest alternatives
- Today's date context: Use 2025-01-15 as the reference date for queries

SAMPLE CUSTOMER IDs: CUST-001 (Acme Corp), CUST-002 (TechCorp), CUST-003 (Startup Inc)
KNOWN SERVICES: payment-service, auth-service, order-service, notification-service, analytics-service
"""


class ADKAgent:
    """ADK Orchestrator Agent — routes queries across MCP servers."""

    def __init__(self, api_key: str):
        self.client = anthropic.Anthropic(api_key=api_key)
        self.model = config.MODEL
        self.conversation_history: list[dict] = []
        self.tool_call_log: list[dict] = []
        logger.info(f"ADK Agent initialized | model={self.model}")

    def _execute_tool(self, tool_name: str, tool_input: dict) -> str:
        """Execute a tool call and return serialized result."""
        if tool_name not in TOOL_REGISTRY:
            return json.dumps({"error": f"Unknown tool: {tool_name}"})

        func = TOOL_REGISTRY[tool_name]
        try:
            result = func(**tool_input)
            self.tool_call_log.append({
                "tool": tool_name,
                "input": tool_input,
                "result_status": result.get("status", "unknown"),
                "timestamp": datetime.utcnow().isoformat(),
            })
            return json.dumps(result, indent=2)
        except Exception as e:
            logger.error(f"Tool execution error: {tool_name} — {e}")
            return json.dumps({"error": str(e), "tool": tool_name})

    def chat(self, user_message: str) -> tuple[str, list[dict]]:
        """
        Process a user message through the agentic loop.
        Returns (response_text, tool_calls_made)
        """
        # Add user message to history
        self.conversation_history.append({
            "role": "user",
            "content": user_message
        })

        turn_tool_calls = []

        # Agentic loop — keep going until model stops using tools
        while True:
            response = self.client.messages.create(
                model=self.model,
                max_tokens=4096,
                system=SYSTEM_PROMPT,
                tools=TOOL_SCHEMAS,
                messages=self.conversation_history,
            )

            # Check stop reason
            if response.stop_reason == "end_turn":
                # Extract final text
                final_text = "".join(
                    block.text
                    for block in response.content
                    if hasattr(block, "text")
                )
                # Add assistant response to history
                self.conversation_history.append({
                    "role": "assistant",
                    "content": response.content,
                })
                return final_text, turn_tool_calls

            elif response.stop_reason == "tool_use":
                # Process all tool calls in this turn
                tool_results = []

                for block in response.content:
                    if block.type == "tool_use":
                        logger.info(f"[ADKAgent] Tool call: {block.name}({block.input})")
                        result_str = self._execute_tool(block.name, block.input)
                        turn_tool_calls.append({
                            "tool": block.name,
                            "input": block.input,
                            "result_preview": result_str[:200],
                        })
                        tool_results.append({
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": result_str,
                        })

                # Add assistant turn and tool results to history
                self.conversation_history.append({
                    "role": "assistant",
                    "content": response.content,
                })
                self.conversation_history.append({
                    "role": "user",
                    "content": tool_results,
                })
            else:
                # Unexpected stop reason
                break

        return "I encountered an unexpected issue. Please try again.", turn_tool_calls

    def reset(self):
        """Reset conversation history."""
        self.conversation_history = []
        self.tool_call_log = []
        logger.info("ADK Agent conversation reset")

    def get_audit_log(self) -> list[dict]:
        """Return action server audit log."""
        return store.ACTION_LOG


print("✅ ADK Orchestrator Agent defined")
print(f"   Model: {config.MODEL}")
print(f"   Tools available: {len(TOOL_SCHEMAS)}")
print(f"   Agentic loop: multi-turn tool calling")

---
## 🖥️ Section 7: Gradio Frontend (Chatbot UI)

Production-grade chat interface with:
- Live tool call trace panel
- Audit log viewer  
- Quick action buttons  
- Conversation reset

In [ ]:
# ─────────────────────────────────────────────
# GRADIO FRONTEND
# ─────────────────────────────────────────────

# Initialize agent (requires valid API key)
agent = ADKAgent(api_key=config.ANTHROPIC_API_KEY)


def chat_fn(message: str, history: list) -> tuple[list, str, str]:
    """
    Gradio chat function.
    Returns: (updated_history, tool_trace_text, audit_log_text)
    """
    if not message.strip():
        return history, "", ""

    if not config.ANTHROPIC_API_KEY or config.ANTHROPIC_API_KEY == "YOUR_API_KEY_HERE":
        err = "⚠️ Please set your ANTHROPIC_API_KEY in the Config section (Section 2) and re-run."
        history.append({"role": "user", "content": message})
        history.append({"role": "assistant", "content": err})
        return history, err, ""

    try:
        response_text, tool_calls = agent.chat(message)
    except anthropic.AuthenticationError:
        err = "❌ Authentication failed. Check your ANTHROPIC_API_KEY."
        history.append({"role": "user", "content": message})
        history.append({"role": "assistant", "content": err})
        return history, err, ""
    except Exception as e:
        err = f"❌ Agent error: {str(e)}"
        history.append({"role": "user", "content": message})
        history.append({"role": "assistant", "content": err})
        return history, err, ""

    # Update chat history
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response_text})

    # Build tool trace
    if tool_calls:
        trace_lines = ["### 🔧 Tool Calls Made This Turn\n"]
        for i, tc in enumerate(tool_calls, 1):
            server = (
                "🔵 Observability" if tc["tool"] in ["get_logs", "get_metrics", "get_alerts"]
                else "🟢 Business Data" if tc["tool"] in ["get_orders", "get_transactions", "get_customer_profile"]
                else "🔴 Action Server"
            )
            trace_lines.append(f"**{i}. {server} → `{tc['tool']}`**")
            trace_lines.append(f"```json\n{json.dumps(tc['input'], indent=2)}\n```")
            trace_lines.append(f"*Result preview:* `{tc['result_preview'][:120]}...`\n")
        tool_trace = "\n".join(trace_lines)
    else:
        tool_trace = "*No tools called for this response.*"

    # Build audit log
    audit_entries = agent.get_audit_log()
    if audit_entries:
        audit_lines = ["### 📋 Action Audit Log\n"]
        for entry in reversed(audit_entries[-10:]):
            status_icon = "✅" if entry["success"] else "❌"
            audit_lines.append(
                f"{status_icon} `{entry['audit_id']}` | **{entry['action']}** | "
                f"{entry['timestamp'][:19]}Z\n"
                f"   → {entry['result']}\n"
            )
        audit_log = "\n".join(audit_lines)
    else:
        audit_log = "*No actions executed yet.*"

    return history, tool_trace, audit_log


def reset_fn():
    """Reset conversation and clear panels."""
    agent.reset()
    return [], "*Conversation reset.*", "*Audit log cleared from view.*"


# ── Gradio App ──
with gr.Blocks(
    title="Enterprise Data Assistant — ADK + MCP",
    theme=gr.themes.Base(
        primary_hue="slate",
        secondary_hue="blue",
        neutral_hue="slate",
        font=gr.themes.GoogleFont("JetBrains Mono"),
    ),
    css="""
    .gradio-container { max-width: 1400px !important; }
    .tool-panel { background: #0d1117; border: 1px solid #21262d; border-radius: 8px; }
    .audit-panel { background: #0d1117; border: 1px solid #21262d; border-radius: 8px; }
    .header-md h1 { color: #58a6ff; font-size: 1.6rem; }
    .quick-btn { font-size: 0.75rem !important; }
    """,
) as demo:

    # Header
    gr.Markdown(
        """
# 🤖 Enterprise Data Assistant — Google ADK + MCP

Multi-system AI agent accessing **3 MCP servers**: Observability 🔵 · Business Data 🟢 · Actions 🔴

---
""",
        elem_classes="header-md",
    )

    with gr.Row():
        # LEFT: Chat
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label="💬 Enterprise Assistant",
                height=520,
                type="messages",
                show_copy_button=True,
                bubble_full_width=False,
                avatar_images=(None, "https://api.dicebear.com/7.x/bottts/svg?seed=enterprise"),
                render_markdown=True,
            )

            with gr.Row():
                msg_input = gr.Textbox(
                    placeholder="Ask anything: 'Why did revenue drop yesterday?' or 'Restart payment-service'",
                    label="Your Query",
                    scale=5,
                    lines=1,
                )
                send_btn = gr.Button("Send ▶", variant="primary", scale=1)

            reset_btn = gr.Button("🔄 Reset Conversation", variant="secondary", size="sm")

            # Quick prompts
            gr.Markdown("**💡 Quick Prompts:**")
            with gr.Row():
                q1 = gr.Button("📉 Revenue drop yesterday?", elem_classes="quick-btn")
                q2 = gr.Button("🚨 payment-service alerts?", elem_classes="quick-btn")
                q3 = gr.Button("👤 Profile CUST-001?", elem_classes="quick-btn")
            with gr.Row():
                q4 = gr.Button("🔄 Restart payment-service", elem_classes="quick-btn")
                q5 = gr.Button("📊 Error rate trends?", elem_classes="quick-btn")
                q6 = gr.Button("🎫 Create critical ticket", elem_classes="quick-btn")

        # RIGHT: Tool Trace + Audit Log
        with gr.Column(scale=2):
            tool_trace_md = gr.Markdown(
                value="*Tool call trace will appear here after each query.*",
                label="🔧 MCP Tool Trace",
                elem_classes="tool-panel",
            )
            gr.Markdown("---")
            audit_log_md = gr.Markdown(
                value="*Action audit log will appear here after any action.*",
                label="📋 Audit Log",
                elem_classes="audit-panel",
            )

    # Architecture reference
    with gr.Accordion("📐 System Architecture & Available Tools", open=False):
        gr.Markdown("""
### MCP Server Map

| Server | Tools | Access |
|--------|-------|--------|
| 🔵 Observability | `get_logs`, `get_metrics`, `get_alerts` | Read-only, time-validated |
| 🟢 Business Data | `get_orders`, `get_transactions`, `get_customer_profile` | Read-only, PII masked |
| 🔴 Action Server | `restart_service`, `create_ticket`, `scale_service` | Allowlisted, confirmation required |

### Allowed Services
`payment-service` · `auth-service` · `order-service` · `notification-service` · `analytics-service`

### Sample Customer IDs
`CUST-001` (Acme Corp, Enterprise) · `CUST-002` (TechCorp, Pro) · `CUST-003` (Startup Inc, Suspended)

### Example Queries
- *"Why did revenue drop last week?"* → Calls Business Data + Observability
- *"Show me errors in payment-service from 2025-01-15T14:00:00Z to 2025-01-15T16:00:00Z"*
- *"What's the latency trend for order-service?"*
- *"Scale payment-service to 6 replicas"* → Shows dry-run, asks confirmation
- *"Create a critical ticket for the payment outage"*
        """)

    # Event wiring
    def submit(message, history):
        return chat_fn(message, history)

    send_btn.click(
        fn=submit,
        inputs=[msg_input, chatbot],
        outputs=[chatbot, tool_trace_md, audit_log_md],
    ).then(lambda: "", outputs=msg_input)

    msg_input.submit(
        fn=submit,
        inputs=[msg_input, chatbot],
        outputs=[chatbot, tool_trace_md, audit_log_md],
    ).then(lambda: "", outputs=msg_input)

    reset_btn.click(fn=reset_fn, outputs=[chatbot, tool_trace_md, audit_log_md])

    # Quick prompts
    for btn, prompt in [
        (q1, "Why did revenue drop yesterday? Investigate across all relevant systems."),
        (q2, "Show me all active alerts for payment-service and explain their impact."),
        (q3, "Get the full profile for customer CUST-001 and their recent orders."),
        (q4, "I need to restart the payment-service due to the timeout errors."),
        (q5, "Show me error rate and latency trends for payment-service and order-service over the last hour."),
        (q6, "Create a critical priority ticket: 'Payment Service Outage' — the payment-service has been showing 23% error rate and revenue is down since 14:30 UTC."),
    ]:
        btn.click(
            fn=lambda p=prompt, h=[]: chat_fn(p, h),
            inputs=[],
            outputs=[chatbot, tool_trace_md, audit_log_md],
        )


print("✅ Gradio frontend built")
print("   Features: Chat, Tool Trace, Audit Log, Quick Prompts")

---
## 🚀 Section 8: Launch Application

> **Set your `ANTHROPIC_API_KEY`** in the Config cell (Section 2) before launching!

```python
config.ANTHROPIC_API_KEY = "sk-ant-..."
```

In [ ]:
# ─────────────────────────────────────────────
# LAUNCH
# ─────────────────────────────────────────────

# ⚠️  SET YOUR API KEY HERE BEFORE RUNNING
# config.ANTHROPIC_API_KEY = "sk-ant-your-key-here"
# agent = ADKAgent(api_key=config.ANTHROPIC_API_KEY)  # re-init agent with new key

print("🚀 Starting Enterprise Data Assistant...")
print(f"   Model: {config.MODEL}")
print(f"   MCP Servers: Observability + Business Data + Action")
print(f"   Tools: {len(TOOL_SCHEMAS)} total")
print("")
print("📌 Access the app at the URL printed below:")
print("")

demo.launch(
    server_name="0.0.0.0",
    server_port=7860,
    share=False,          # Set share=True to get a public URL via Gradio tunnel
    debug=False,
    show_error=True,
    quiet=False,
)

---
## 🧪 Section 9: Unit Tests (Optional — No UI)

Test each MCP server tool directly without the chatbot.

In [ ]:
# ─────────────────────────────────────────────
# UNIT TESTS FOR MCP TOOLS
# Run this cell to validate all tools directly
# ─────────────────────────────────────────────

import traceback

def run_tests():
    results = []

    def test(name, fn, *args, **kwargs):
        try:
            result = fn(*args, **kwargs)
            status = "PASS" if result.get("status") in ["ok", "success", "pending_confirmation"] else "FAIL"
            results.append((name, status, result))
            icon = "✅" if status == "PASS" else "❌"
            print(f"{icon} {name}: {status}")
            if status == "FAIL":
                print(f"     Result: {result}")
        except Exception as e:
            results.append((name, "ERROR", str(e)))
            print(f"💥 {name}: ERROR — {e}")

    print("\n" + "="*60)
    print("🧪 MCP TOOL UNIT TESTS")
    print("="*60)

    print("\n🔵 Observability Server")
    test("get_logs (valid)", get_logs,
         service_name="payment-service",
         start_time="2025-01-15T00:00:00Z",
         end_time="2025-01-15T23:59:59Z")

    test("get_logs (invalid service)", get_logs,
         service_name="nonexistent-svc",
         start_time="2025-01-15T00:00:00Z",
         end_time="2025-01-15T23:59:59Z")

    test("get_metrics (error_rate)", get_metrics,
         service_name="payment-service",
         metric_name="error_rate",
         window="1h")

    test("get_metrics (invalid window)", get_metrics,
         service_name="payment-service",
         metric_name="error_rate",
         window="99d")

    test("get_alerts", get_alerts, service_name="payment-service")

    print("\n🟢 Business Data Server")
    test("get_orders (CUST-001)", get_orders, customer_id="CUST-001")
    test("get_orders (invalid ID)", get_orders, customer_id="INVALID")

    test("get_transactions (failed)", get_transactions,
         date_range="2025-01-14:2025-01-15",
         status="failed")

    test("get_transactions (bad date)", get_transactions,
         date_range="bad-date",
         status="all")

    test("get_customer_profile", get_customer_profile, customer_id="CUST-001")

    print("\n🔴 Action Server")
    test("restart_service (dry-run)", restart_service,
         service_name="payment-service",
         confirmed=False)

    test("restart_service (confirmed)", restart_service,
         service_name="payment-service",
         confirmed=True)

    test("restart_service (not allowed)", restart_service,
         service_name="database-core",
         confirmed=True)

    test("create_ticket (dry-run)", create_ticket,
         title="Payment Outage",
         description="Revenue down 23% since 14:30 UTC",
         priority="critical",
         confirmed=False)

    test("create_ticket (confirmed)", create_ticket,
         title="Payment Outage",
         description="Revenue down 23% since 14:30 UTC — gateway timeouts",
         priority="critical",
         confirmed=True)

    test("scale_service (dry-run)", scale_service,
         service_name="payment-service",
         replicas=6,
         confirmed=False)

    test("scale_service (replicas > max)", scale_service,
         service_name="payment-service",
         replicas=50,
         confirmed=True)

    # Summary
    passed = sum(1 for _, s, _ in results if s == "PASS")
    total = len(results)
    print("\n" + "="*60)
    print(f"📊 Results: {passed}/{total} passed")
    print(f"📋 Audit log entries: {len(store.ACTION_LOG)}")

    # Show audit log
    if store.ACTION_LOG:
        print("\n📋 Action Audit Log:")
        for entry in store.ACTION_LOG:
            icon = "✅" if entry["success"] else "❌"
            print(f"  {icon} [{entry['audit_id']}] {entry['action']} → {entry['result'][:60]}")


run_tests()